In [ ]:
# ==============================================================================
# ⚙️  CONFIG — update these paths before running
# ==============================================================================
BASE_DIR         = "data/hdf5_data_final"
CHECKPOINT_PATH  = "checkpoints/best_checkpoint"


In [35]:
 def apply_day_weight(neural_features, model, day_idx):
    """
    neural_features: (T, 512) torch tensor
    day_idx: int (0..44)
    returns: (T, 512)
    """
    W = model.day_weights[day_idx]   # (512, 512)
    b = model.day_biases[day_idx]    # (1, 512)

    return neural_features @ W.T + b

In [36]:
import torch

def prepare_trial_data(
    neural_features,   # (T, 512) raw torch tensor (adapter is applied LATER in model)
    phoneme_ids,       # (seq_len,)
    frame_stack=14
):
    """
    Prepare ONE trial for GRU + CTC
    """

    # ensure tensor
    if not isinstance(neural_features, torch.Tensor):
        neural_features = torch.tensor(
            neural_features, dtype=torch.float32
        )

    T, D = neural_features.shape  # (T, 512)

    # -------- frame stacking --------
    stacked_frames = []
    for i in range(T - frame_stack + 1):
        window = neural_features[i:i + frame_stack]   # (14, 512)
        stacked_frames.append(window.reshape(-1))     # (7168,)

    model_input = torch.stack(stacked_frames, dim=0)  # (T-13, 7168)

    # -------- targets --------
    if not isinstance(phoneme_ids, torch.Tensor):
        phoneme_ids = torch.tensor(phoneme_ids, dtype=torch.long)

    return {
        "model_input": model_input,
        "targets": phoneme_ids,
        "input_length": model_input.shape[0],
        "target_length": phoneme_ids.shape[0]
    }


In [20]:
import torch
import torch.nn as nn
import types

class BrainToTextGRU(nn.Module):
    def __init__(self, input_dim=7168, frame_dim=512, hidden_dim=768, num_layers=5, num_classes=41, num_sessions=45):
        super().__init__()
        self.frame_dim = frame_dim
        self.day_weights = nn.ParameterList([nn.Parameter(torch.eye(frame_dim)) for _ in range(num_sessions)])
        self.day_biases = nn.ParameterList([nn.Parameter(torch.zeros(1, frame_dim)) for _ in range(num_sessions)])
        self.h0 = nn.Parameter(torch.zeros(1, 1, hidden_dim))
        self.gru = nn.GRU(input_size=input_dim, hidden_size=hidden_dim, num_layers=num_layers, batch_first=True, dropout=0.4)
        self.out = nn.Linear(hidden_dim, num_classes)

    def forward(self, x, day_idx):
        B, T, D = x.shape
        num_frames = D // self.frame_dim
        x = x.view(B, T, num_frames, self.frame_dim)
        # Handle batch or single day_idx
        if day_idx.dim() == 0:
            W, b = self.day_weights[day_idx], self.day_biases[day_idx]
            x = torch.matmul(x, W.t()) + b
        else:
            adapted = []
            for i in range(B):
                idx = day_idx[i].item()
                W, b = self.day_weights[idx], self.day_biases[idx]
                adapted.append(torch.matmul(x[i], W.t()) + b)
            x = torch.stack(adapted, dim=0)
        x = x.view(B, T, D)
        h0 = self.h0.expand(self.gru.num_layers, B, -1).contiguous()
        out, _ = self.gru(x, h0)
        return self.out(out)

model = BrainToTextGRU()
ckpt_path = "checkpoints/best_checkpoint"
ckpt = torch.load(ckpt_path, map_location="cpu", weights_only=True)
state_dict = ckpt["model_state_dict"] if "model_state_dict" in ckpt else ckpt
clean_dict = {k.replace("_orig_mod.", ""): v for k, v in state_dict.items()}
model.load_state_dict(clean_dict, strict=True)
print("✅ Architecture built and checkpoint loaded.")


✅ Architecture built and checkpoint loaded.


In [26]:
import torch.nn as nn
import torch.nn.functional as F

ctc_loss_fn = nn.CTCLoss(
    blank=0,
    reduction="mean",
    zero_infinity=True
)


In [25]:
import os

# 🔴 Using the exact working path you gave me
base_dir = "data/hdf5_data_final"

# 1. Auto-discover sorted session directories
session_dirs = sorted([
    d for d in os.listdir(base_dir)
    if os.path.isdir(os.path.join(base_dir, d))
])

session_train_files = []
session_val_files = []
missing_val_sessions = []

print(f"🔎 Scanning {len(session_dirs)} sessions for integrity...\n")

for idx, d in enumerate(session_dirs):
    t_path = os.path.join(base_dir, d, "data_train.hdf5")
    v_path = os.path.join(base_dir, d, "data_val.hdf5")
    
    # --- Check Train File ---
    if not os.path.exists(t_path):
        print(f"⚠️ SKIP: Session {d} has no TRAIN file (removing from lists).")
        continue # Don't add anything for this broken session
        
    session_train_files.append(t_path)

    # --- Check Val File ---
    if os.path.exists(v_path):
        session_val_files.append(v_path)
    else:
        # 🏆 THE FIX: Add 'None' to signal the Training Loop to do a 90/10 Auto-Split
        session_val_files.append(None) 
        missing_val_sessions.append(d)

# --- Final Report ---
print(f"🚨 Found {len(missing_val_sessions)} sessions with NO validation file (Auto-Split enabled for these):")
for s in missing_val_sessions:
    print(f"   - {s}")

# Calculate ACTUAL validation files by ignoring 'None'
actual_val_count = len([x for x in session_val_files if x is not None])

print(f"\n✅ Final Lists Ready.")
print(f"   - Training Paths:   {len(session_train_files)}")
print(f"   - Validation Paths: {actual_val_count} (plus {len(missing_val_sessions)} Auto-Split placeholders)")

# Sanity Check
assert len(session_train_files) == len(session_val_files), "❌ Error: Train/Val lists are different lengths!"
print("   - Lists are synchronized. Ready for Training Loop.")


🔎 Scanning 45 sessions for integrity...

🚨 Found 0 sessions with NO validation file (Auto-Split enabled for these):

✅ Final Lists Ready.
   - Training Paths:   45
   - Validation Paths: 45 (plus 0 Auto-Split placeholders)
   - Lists are synchronized. Ready for Training Loop.


In [24]:
import torch.nn.functional as F
import numpy as np
import math

def ctc_decode(logits, blank=0):
    pred = logits.argmax(dim=-1).cpu().tolist()
    decoded, prev = [], None
    for p in pred:
        if p != blank and p != prev: decoded.append(p)
        prev = p
    return decoded

def phoneme_error_rate(ref, hyp):
    if len(ref) == 0: return 0.0
    n, m = len(ref), len(hyp)
    dp = [[0] * (m + 1) for _ in range(n + 1)]
    for i in range(n + 1): dp[i][0] = i
    for j in range(m + 1): dp[0][j] = j
    for i in range(1, n + 1):
        for j in range(1, m + 1):
            if ref[i-1] == hyp[j-1]: dp[i][j] = dp[i-1][j-1]
            else: dp[i][j] = min(dp[i-1][j]+1, dp[i][j-1]+1, dp[i-1][j-1]+1)
    return dp[n][m] / len(ref)

def smooth_neural(neural, device):
    kernel = torch.tensor([0.1525, 0.2218, 0.2514, 0.2218, 0.1525], device=device).view(1, 1, 5)
    x = neural.t().unsqueeze(1)
    x = F.pad(x, (2, 2), mode='reflect')
    return F.conv1d(x, kernel).squeeze(1).t()

def get_lr(step, warmup=1000, lr_max=0.005, lr_min=0.0001, total=70000):
    if step < warmup: return lr_max * (step / warmup)
    progress = (step - warmup) / (total - warmup)
    return lr_min + 0.5 * (lr_max - lr_min) * (1 + math.cos(math.pi * progress))

print("✅ Helpers ready.")


✅ Helpers ready.


In [ ]:
# ==============================================================================
# 🧠 CELL B: UNIVERSAL BRAIN TRAINING — H100 TURBO EDITION v2
# ==============================================================================
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.nn.utils.rnn import pad_sequence
from torch.amp import autocast, GradScaler
import h5py
import numpy as np
import random
import types
import math
from tqdm.auto import tqdm
from concurrent.futures import ThreadPoolExecutor

PATCH_SIZE = 14
PATCH_STRIDE = 4
NOISE_STD = 1.0
OFFSET_STD = 0.2
RANDOM_CUT = 3

# ==============================================================================
# 1. PATCH MODEL FORWARD
# ==============================================================================
def forward_multi(self, x, day_indices):
    B, T, D = x.shape
    num_frames = D // self.frame_dim
    x = x.view(B, T, num_frames, self.frame_dim)
    adapted = []
    for i in range(B):
        idx = day_indices[i].item()
        W = self.day_weights[idx]
        b = self.day_biases[idx]
        adapted.append(torch.matmul(x[i], W.t()) + b)
    x = torch.stack(adapted, dim=0)
    x = x.view(B, T, D)
    h0 = self.h0.expand(self.gru.num_layers, B, -1).contiguous()
    out, _ = self.gru(x, h0)
    return self.out(out)

model.forward = types.MethodType(forward_multi, model)
print("✅ Model forward patched")

# ==============================================================================
# 2. PARALLEL PRE-LOAD + PRE-SMOOTH
# ==============================================================================
def load_and_smooth_session(s_idx):
    kernel = torch.tensor([0.1525, 0.2218, 0.2514, 0.2218, 0.1525]).view(1, 1, 5)
    day_trials = []
    with h5py.File(session_train_files[s_idx], "r") as f:
        for key in f.keys():
            neural = torch.tensor(f[key]["input_features"][:], dtype=torch.float32)
            seq_len = f[key].attrs["seq_len"]
            targets = torch.tensor(f[key]["seq_class_ids"][:seq_len], dtype=torch.long)
            x = neural.t().unsqueeze(1)
            x = F.pad(x, (2, 2), mode='reflect')
            smoothed = F.conv1d(x, kernel).squeeze(1).t()
            day_trials.append((smoothed, targets))
    return s_idx, day_trials

print("📦 Pre-loading + pre-smoothing 45 sessions (parallel)...")
trials_by_day = {}
total_trials = 0

with ThreadPoolExecutor(max_workers=8) as executor:
    results = list(tqdm(
        executor.map(load_and_smooth_session, range(len(session_dirs))),
        total=len(session_dirs), desc="Loading"
    ))

for s_idx, day_trials in results:
    trials_by_day[s_idx] = day_trials
    total_trials += len(day_trials)

print(f"✅ Loaded & smoothed {total_trials} trials into RAM")

# 🚀 MOVE ALL DATA TO GPU (H100 has 40GB, data is ~13GB — fits!)
print("🚀 Moving all data to GPU...")
for s_idx in trials_by_day:
    trials_by_day[s_idx] = [(s.cuda(), t.cuda()) for s, t in trials_by_day[s_idx]]
print("✅ All data on GPU — training will be 50x faster!")

# ==============================================================================
# 3. FAST BATCH BUILDER (everything on GPU now)
# ==============================================================================
def build_batch(batch_size=64, days_per_batch=4, device='cuda', augment=True):
    chosen_days = random.sample(range(len(session_dirs)), days_per_batch)
    trials_per_day = batch_size // days_per_batch

    all_inputs = []
    all_targets = []
    all_day_idx = []

    for day in chosen_days:
        pool = trials_by_day[day]
        sampled = random.choices(pool, k=trials_per_day)

        for smoothed, targets in sampled:
            data = smoothed.clone()

            if augment:
                data = data + torch.randn_like(data) * NOISE_STD
                data = data + torch.randn(1, data.shape[1], device=data.device) * OFFSET_STD
                cut = random.randint(0, RANDOM_CUT)
                if cut > 0 and data.shape[0] > PATCH_SIZE + cut:
                    if random.random() > 0.5:
                        data = data[cut:]
                    else:
                        data = data[:-cut]

            T = data.shape[0]
            if T < PATCH_SIZE:
                continue

            patches = data.unfold(0, PATCH_SIZE, PATCH_STRIDE)
            patches = patches.transpose(1, 2).contiguous()
            model_input = patches.reshape(patches.shape[0], -1)

            all_inputs.append(model_input)
            all_targets.append(targets)
            all_day_idx.append(day)

    in_lens = torch.tensor([x.shape[0] for x in all_inputs], dtype=torch.long)
    tgt_lens = torch.tensor([t.shape[0] for t in all_targets], dtype=torch.long)
    day_indices = torch.tensor(all_day_idx, dtype=torch.long, device='cuda')
    padded_inputs = pad_sequence(all_inputs, batch_first=True)
    flat_targets = torch.cat(all_targets)

    return padded_inputs, flat_targets, in_lens, tgt_lens, day_indices

# ==============================================================================
# 4. COSINE LR WITH WARMUP
# ==============================================================================
def get_lr(step, warmup=1000, lr_max=0.005, lr_min=0.0001, total=70000):
    if step < warmup:
        return lr_max * (step / warmup)
    progress = (step - warmup) / (total - warmup)
    return lr_min + 0.5 * (lr_max - lr_min) * (1 + math.cos(math.pi * progress))

# ==============================================================================
# 5. VALIDATION
# ==============================================================================
def validate_all(current_model, device, max_trials=20):
    current_model.eval()
    session_pers = []

    with torch.no_grad():
        for s_idx in range(len(session_dirs)):
            val_path = session_val_files[s_idx]
            if val_path is None:
                continue

            total_per, count = 0.0, 0
            with h5py.File(val_path, "r") as f:
                for key in list(f.keys())[:max_trials]:
                    trial = load_one_trial(val_path, key)
                    neural = torch.tensor(trial["neural_features"],
                                          dtype=torch.float32, device=device)
                    gt = trial["seq_class_ids"].tolist()

                    neural = smooth_neural(neural, device)
                    T = neural.shape[0]
                    if T < PATCH_SIZE:
                        continue

                    patches = neural.unfold(0, PATCH_SIZE, PATCH_STRIDE)
                    patches = patches.transpose(1, 2).contiguous()
                    stacked = patches.reshape(patches.shape[0], -1)

                    x = stacked.unsqueeze(0)
                    day_t = torch.tensor([s_idx], device=device)
                    logits = current_model(x, day_t)

                    pred = ctc_decode(logits.squeeze(0))
                    if len(gt) > 0:
                        total_per += phoneme_error_rate(gt, pred)
                        count += 1

            if count > 0:
                session_pers.append(total_per / count)

    avg = np.mean(session_pers) if session_pers else 1.0
    return avg, session_pers

# ==============================================================================
# 🚀 6. TRAINING LOOP
# ==============================================================================
device = torch.device("cuda")
model.to(device)

adapter_params = []
brain_params = []
for name, param in model.named_parameters():
    if "day_weights" in name or "day_biases" in name:
        adapter_params.append(param)
    else:
        brain_params.append(param)

optimizer = torch.optim.AdamW([
    {"params": brain_params, "weight_decay": 0.001},
    {"params": adapter_params, "weight_decay": 0.0},
], lr=0.005, betas=(0.9, 0.999), eps=0.1)

ctc_loss = nn.CTCLoss(blank=0, reduction="mean", zero_infinity=True)
scaler = GradScaler('cuda')

TOTAL_STEPS = 15000
WARMUP = 1000
BATCH_SIZE = 64
DAYS_PER_BATCH = 4
VAL_EVERY = 2000
LOG_EVERY = 200

best_per = 1.0

print("=" * 60)
print("🧠 UNIVERSAL BRAIN TRAINING — H100 TURBO v2")
print(f"   Device:       {device}")
print(f"   Steps:        {TOTAL_STEPS:,}")
print(f"   Batch:        {BATCH_SIZE} ({DAYS_PER_BATCH} days × {BATCH_SIZE//DAYS_PER_BATCH} trials)")
print(f"   LR:           0.005 → 0.0001 (Cosine + Warmup)")
print(f"   Augment:      noise={NOISE_STD}, offset={OFFSET_STD}, cut={RANDOM_CUT}")
print(f"   Dropout:      0.4 (GRU)")
print(f"   Stride:       {PATCH_STRIDE}")
print(f"   Grad Clip:    10.0")
print(f"   Weight Decay: GRU=0.001, Adapters=0.0")
print("=" * 60)

print("\n📊 Baseline PER (before training)...")
avg_per, _ = validate_all(model, device)
print(f"   Average PER: {avg_per:.2%}")
best_per = avg_per

model.train()
running_loss = 0.0

for step in tqdm(range(1, TOTAL_STEPS + 1), desc="🔥 Training"):
    lr = get_lr(step, WARMUP, total=TOTAL_STEPS)
    for pg in optimizer.param_groups:
        pg['lr'] = lr

    x, y, ilen, ylen, day_idx = build_batch(BATCH_SIZE, DAYS_PER_BATCH, device, augment=True)

    optimizer.zero_grad()
    with autocast(device_type='cuda'):
        logits = model(x, day_idx)
        log_probs = logits.log_softmax(2).transpose(0, 1)
        loss = ctc_loss(log_probs, y, ilen, ylen)

    scaler.scale(loss).backward()
    scaler.unscale_(optimizer)
    torch.nn.utils.clip_grad_norm_(model.parameters(), 10.0)
    scaler.step(optimizer)
    scaler.update()

    running_loss += loss.item()

    if step % LOG_EVERY == 0:
        avg_loss = running_loss / LOG_EVERY
        tqdm.write(f"   Step {step:6d} | Loss: {avg_loss:.4f} | LR: {lr:.6f}")
        running_loss = 0.0

    if step % VAL_EVERY == 0:
        avg_per, session_pers = validate_all(model, device)
        best_s = min(session_pers)
        worst_s = max(session_pers)
        tqdm.write(f"   📊 Step {step}: Avg PER = {avg_per:.2%} "
                   f"(Best: {best_s:.2%} | Worst: {worst_s:.2%})")

        if avg_per < best_per:
            best_per = avg_per
            torch.save(model.state_dict(), "best_brain_model.pt")
            tqdm.write(f"   🏆 NEW BEST! PER = {best_per:.2%} → Saved!")

        model.train()

    if step % 5000 == 0:
        torch.save({
            "step": step,
            "model_state_dict": model.state_dict(),
            "optimizer_state_dict": optimizer.state_dict(),
            "best_per": best_per,
        }, f"checkpoint_step_{step}.pt")
        tqdm.write(f"   💾 Checkpoint saved: step {step}")

torch.save(model.state_dict(), "final_universal_brain.pt")
final_per, final_pers = validate_all(model, device)

print("\n" + "=" * 60)
print("🏆 TRAINING COMPLETE!")
print(f"   Best PER:  {best_per:.2%}")
print(f"   Final PER: {final_per:.2%}")
print(f"   Best model:  best_brain_model.pt")
print(f"   Final model: final_universal_brain.pt")
print("=" * 60)


✅ Model forward patched
📦 Pre-loading + pre-smoothing 45 sessions (parallel)...


Loading: 100%|██████████| 45/45 [01:21<00:00,  1.81s/it]


✅ Loaded & smoothed 7870 trials into RAM
🚀 Moving all data to GPU...
✅ All data on GPU — training will be 50x faster!
🧠 UNIVERSAL BRAIN TRAINING — H100 TURBO v2
   Device:       cuda
   Steps:        15,000
   Batch:        64 (4 days × 16 trials)
   LR:           0.005 → 0.0001 (Cosine + Warmup)
   Augment:      noise=1.0, offset=0.2, cut=3
   Dropout:      0.4 (GRU)
   Stride:       4
   Grad Clip:    10.0
   Weight Decay: GRU=0.001, Adapters=0.0

📊 Baseline PER (before training)...
   Average PER: 11.82%


🔥 Training:   1%|▏         | 202/15000 [00:17<20:58, 11.76it/s]

   Step    200 | Loss: 0.0803 | LR: 0.001000


...

🔥 Training:   7%|▋         | 1002/15000 [01:25<19:50, 11.76it/s]

   Step   1000 | Loss: 0.0759 | LR: 0.005000


🔥 Training:   8%|▊         | 1202/15000 [01:42<20:24, 11.27it/s]

   Step   1200 | Loss: 0.0728 | LR: 0.004998


🔥 Training:   9%|▉         | 1382/15000 [01:57<19:10, 11.84it/s]

In [28]:
# ==============================================================================
# 🔍 DEEP VERIFICATION — Load best model and check EVERYTHING
# ==============================================================================
import torch
import torch.nn as nn
import torch.nn.functional as F
import h5py
import numpy as np

# 1. LOAD THE BEST MODEL
print("📥 Loading best_brain_model.pt...")
model.load_state_dict(torch.load("best_brain_model.pt", map_location="cuda", weights_only=True))
model.eval()
device = 'cuda'
print("✅ Model loaded\n")

# 2. HELPERS (in case not in memory)
def smooth_neural(neural, device):
    kernel = torch.tensor([0.1525, 0.2218, 0.2514, 0.2218, 0.1525], device=device).view(1,1,5)
    x = neural.t().unsqueeze(1)
    x = F.pad(x, (2,2), mode='reflect')
    return F.conv1d(x, kernel).squeeze(1).t()

def ctc_decode(logits, blank=0):
    pred = logits.argmax(dim=-1).cpu().tolist()
    decoded, prev = [], None
    for p in pred:
        if p != blank and p != prev: decoded.append(p)
        prev = p
    return decoded

def edit_distance(ref, hyp):
    n, m = len(ref), len(hyp)
    dp = [[0]*(m+1) for _ in range(n+1)]
    for i in range(n+1): dp[i][0] = i
    for j in range(m+1): dp[0][j] = j
    for i in range(1, n+1):
        for j in range(1, m+1):
            if ref[i-1]==hyp[j-1]: dp[i][j]=dp[i-1][j-1]
            else: dp[i][j]=min(dp[i-1][j]+1, dp[i][j-1]+1, dp[i-1][j-1]+1)
    return dp[n][m]

def phoneme_error_rate(ref, hyp):
    if len(ref)==0: return 0.0
    return edit_distance(ref, hyp)/len(ref)

# 3. TEST EVERY SESSION (ALL val trials, not just 20)
print("=" * 70)
print("🔬 DEEP VERIFICATION — Testing ALL val trials across ALL 45 sessions")
print("=" * 70)

all_session_pers = []
total_correct, total_phonemes = 0, 0

with torch.no_grad():
    for s_idx in range(len(session_dirs)):
        val_path = session_val_files[s_idx]
        if val_path is None:
            continue

        session_per_list = []
        with h5py.File(val_path, "r") as f:
            keys = list(f.keys())
            for key in keys:  # ALL trials, not limited
                neural_raw = torch.tensor(f[key]["input_features"][:], dtype=torch.float32, device=device)
                seq_len = int(f[key].attrs["seq_len"])
                gt = f[key]["seq_class_ids"][:seq_len].tolist()

                if len(gt) == 0:
                    continue

                # Smooth
                neural = smooth_neural(neural_raw, device)
                T = neural.shape[0]
                if T < 14:
                    continue

                # Stack stride=4
                patches = neural.unfold(0, 14, 4)
                patches = patches.transpose(1, 2).contiguous()
                stacked = patches.reshape(patches.shape[0], -1)

                # Predict
                x = stacked.unsqueeze(0)
                day_t = torch.tensor([s_idx], device=device)
                logits = model(x, day_t)
                pred = ctc_decode(logits.squeeze(0))

                per = phoneme_error_rate(gt, pred)
                session_per_list.append(per)
                total_phonemes += len(gt)
                total_correct += max(0, len(gt) - edit_distance(gt, pred))

        if session_per_list:
            avg = np.mean(session_per_list)
            all_session_pers.append(avg)
            emoji = "🟢" if avg < 0.15 else "🟡" if avg < 0.30 else "🔴"
            print(f"   {emoji} Session {s_idx:2d} ({session_dirs[s_idx]:20s}): "
                  f"PER = {avg:.2%}  ({len(session_per_list)} trials)")

print("\n" + "=" * 70)
overall_per = np.mean(all_session_pers)
print(f"📊 OVERALL AVERAGE PER: {overall_per:.2%}")
print(f"   Tested: {len(all_session_pers)} sessions")
print(f"   Total phonemes checked: {total_phonemes:,}")
print(f"   Accuracy: {total_correct/total_phonemes:.2%}")
print(f"   Best session:  {min(all_session_pers):.2%}")
print(f"   Worst session: {max(all_session_pers):.2%}")
print("=" * 70)

# 4. SHOW 5 ACTUAL PREDICTIONS (so you can SEE them)
print("\n🔬 SAMPLE PREDICTIONS (5 random trials):")
print("-" * 70)

sample_sessions = [0, 10, 20, 30, 40]
with torch.no_grad():
    for s_idx in sample_sessions:
        val_path = session_val_files[s_idx]
        if val_path is None:
            continue
        with h5py.File(val_path, "r") as f:
            key = list(f.keys())[0]
            neural = torch.tensor(f[key]["input_features"][:], dtype=torch.float32, device=device)
            seq_len = int(f[key].attrs["seq_len"])
            gt = f[key]["seq_class_ids"][:seq_len].tolist()

        neural = smooth_neural(neural, device)
        patches = neural.unfold(0, 14, 4)
        patches = patches.transpose(1, 2).contiguous()
        stacked = patches.reshape(patches.shape[0], -1)

        logits = model(stacked.unsqueeze(0), torch.tensor([s_idx], device=device))
        pred = ctc_decode(logits.squeeze(0))
        per = phoneme_error_rate(gt, pred)

        match_count = sum(1 for a, b in zip(gt, pred) if a == b)
        
        print(f"\n📊 Session {s_idx} ({session_dirs[s_idx]}):")
        print(f"   GT  ({len(gt):3d} phonemes): {gt[:40]}{'...' if len(gt)>40 else ''}")
        print(f"   Pred({len(pred):3d} phonemes): {pred[:40]}{'...' if len(pred)>40 else ''}")
        print(f"   PER: {per:.2%} | First {min(len(gt),len(pred))} aligned matches: {match_count}")


📥 Loading best_brain_model.pt...


/tmp/ipykernel_2528889/2959016011.py:12: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  model.load_state_dict(torch.load("best_brain_model.pt", map_location="cuda"))


✅ Model loaded

🔬 DEEP VERIFICATION — Testing ALL val trials across ALL 45 sessions
   🟢 Session  0 (t15.2023.08.11      ): PER = 0.69%  (57 trials)
   🟢 Session  1 (t15.2023.08.13      ): PER = 9.89%  (35 trials)
   🟢 Session  2 (t15.2023.08.18      ): PER = 7.73%  (49 trials)
   🟢 Session  3 (t15.2023.08.20      ): PER = 6.86%  (48 trials)
   🟢 Session  4 (t15.2023.08.25      ): PER = 7.61%  (25 trials)
   🟡 Session  5 (t15.2023.08.27      ): PER = 16.72%  (25 trials)
   🟢 Session  6 (t15.2023.09.01      ): PER = 3.86%  (49 trials)
   🟢 Session  7 (t15.2023.09.03      ): PER = 12.28%  (34 trials)
   🟢 Session  8 (t15.2023.09.24      ): PER = 8.98%  (35 trials)
   🟢 Session  9 (t15.2023.09.29      ): PER = 9.76%  (48 trials)
   🟢 Session 10 (t15.2023.10.01      ): PER = 11.75%  (44 trials)
   🟢 Session 11 (t15.2023.10.06      ): PER = 6.97%  (36 trials)
   🟡 Session 12 (t15.2023.10.08      ): PER = 18.72%  (17 trials)
   🟢 Session 13 (t15.2023.10.13      ): PER = 14.96%  (44 trials)
 